# Notebook : aide à la lecture de schémas électroniques

### Bouts de code potentiellement utiles
Afficher plusieurs images pour que l'utilisateur en choisisse une

In [1]:
import os
import tkinter as tk
from PIL import Image, ImageTk

def afficher_miniatures(dossier):
    # Variable to store the name of the selected image
    selection = {"nom": None}

    # Create the main window
    root = tk.Tk()
    root.title("Choose an item")

    # Frame to contain the thumbnails
    frame = tk.Frame(root)
    frame.pack(padx=10, pady=10)

    # Sorted list of files
    fichiers = sorted(os.listdir(dossier))
    images = []

    for fichier in fichiers:
        chemin = os.path.join(dossier, fichier)
        if os.path.isfile(chemin):
            try:
                # Create the thumbnail
                img = Image.open(chemin)
                img.thumbnail((300, 300))  # thumbnail size
                photo = ImageTk.PhotoImage(img)
                images.append(photo)

                # Clickable button with the image
                btn = tk.Button(frame, image=photo,
                                command=lambda f=fichier: choisir_image(f, root, selection))
                btn.pack(side="left", padx=5, pady=5)
            except Exception as e:
                print(f"Error with {fichier}: {e}")

    root.mainloop()
    return selection["nom"]

def choisir_image(fichier, root, selection):
    selection["nom"] = fichier
    root.destroy()


And the corresponding main function :

In [2]:
#if __name__ == "__main__":
    #dossier = "./notebook_files/IMAGES"
    #image_choisie = afficher_miniatures(dossier)
    #print("Selected image:", image_choisie)

### Permettre à l'utilisateur d'upload directement son document
Permet de ne pas coder le chemin en dur dans les scripts

Doc : 
https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20List.html

In [3]:
import ipywidgets as widgets
from IPython.display import display
import os

uploader = widgets.FileUpload(accept='.pdf', multiple=False)
out = widgets.Output()

def on_upload(change):
    global save_path
    with out:
        out.clear_output()
        if uploader.value:
            print("Le PDF a été pris en compte")

            # Récupération du fichier
            pdf_data = uploader.value[0]['content']
            filename = uploader.value[0]['name']

            # Chemin de sauvegarde
            save_path = os.path.join("uploaded_images", filename)

            # Sauvegarde
            with open(save_path, "wb") as f:
                f.write(pdf_data)

            print(f"PDF enregistré dans : {save_path}")

        else:
            print("Aucun fichier fourni")

uploader.observe(on_upload, names='value')

display(widgets.VBox([uploader, out]))


## Transformer le notebook en application web

Ligne de commande (dans le répertoire du notebook, avec Voila installé):


$ voila NotebookEP.ipynb --Voila.ip=127.0.0.1 --Voila.port=8866

Puis, dans un navigateur :

http://127.0.0.1:8866

# Code assemblé : l'application


Cette partie analyse les fichiers au format PDF pour en extraire automatiquement trois types de zones :

- **Table** : tableau BOM (Bill of Materials), c'est-à-dire la liste des composants
- **Diagram** : schéma électronique
- **Board** : carte PCB (circuit imprimé)

Cette partie est composée de trois fichiers qui s'articulent ensemble :

| Fichier | Rôle |
|---|---|
| `prediction.py` | Script autonome : détecte et sauvegarde toutes les zones d'un PDF |
| `merge.py` | Orchestrateur : sélectionne la meilleure zone par classe et lance l'interface |
| `test1.py` | Interface visuelle interactive : affiche et relie les composants détectés |

**Pipeline global :**

```
PDF → conversion en images (pdf2image) → détection de zones (YOLO best.pt)
    → OCR des étiquettes (Tesseract + YOLO bestr.pt) → interface interactive (Matplotlib)
```

**Deux modèles YOLO distincts sont utilisés :**

| Modèle | Utilisé dans | Ce qu'il détecte |
|---|---|---|
| `best.pt` | `prediction.py`, `merge.py` | Grandes zones dans le PDF : Table, Diagram, Board |
| `bestr.pt` | `test1.py` | Petites étiquettes de composants dans les images (R12, C4, Q1…) |

---
## 1. `prediction.py` — Extraction brute des zones

Le script parcourt toutes les pages d'un PDF, détecte les zones via YOLO et sauvegarde chaque découpe en image PNG dans un dossier de sortie. Il ne filtre pas, ne choisit pas : il extrait tout.

**Usage :**
```bash
python3 prediction.py document.pdf /dossier/sortie/
```

### 1.1 Imports et chargement du modèle

Libraries to install:
- pathlib
- pdf2image
- numpy
- cv2
- ultralytics
- tqdm

On charge le modèle YOLO `best.pt`, entraîné spécifiquement pour reconnaître les trois classes dans des fiches techniques : `Table`, `Diagram`, `Board`.

Les arguments en ligne de commande fournissent le chemin du PDF à analyser et le dossier de destination des images découpées.

In [21]:
import os
import sys
from pathlib import Path
from pdf2image import convert_from_path
import numpy as np
import cv2
from tqdm.notebook import tqdm


from ultralytics import YOLO



# Chargement du modèle entraîné
model_extraction = YOLO("models/extraction/best.pt")

# À adapter selon votre arborescence
pdf_path = {save_path}
dir_path = 'IMAGES'

# Je pense qu'on peut retirer ça
if not os.path.exists(save_path):
    print(f"⚠️ Erreur : Le fichier {pdf_path} est introuvable.")
    
os.makedirs(dir_path, exist_ok=True)


NameError: name 'save_path' is not defined

### 1.2 Conversion du PDF en images

`pdf2image` transforme chaque page du PDF en une image PIL. Le paramètre `dpi=300` garantit une résolution suffisante pour que le texte et les tracés restent lisibles lors de la détection YOLO et de l'OCR qui suivent.

In [ ]:
print("Le traitement est en cours...")
# Conversion du PDF en images (300 DPI)
pages = convert_from_path(save_path, dpi=300)
print(f"{len(pages)} page(s) trouvée(s).")


### 1.3 Détection YOLO et découpe des zones

Pour chaque page, YOLO prédit les boîtes englobantes (`bounding boxes`) avec un seuil de confiance de `0.212` (valeur basse car le modèle est spécialisé et les faux positifs sont rares).

Chaque zone détectée est découpée dans l'image de la page avec une marge de 10 pixels de chaque côté pour ne pas rogner les bords, puis sauvegardée en PNG. Le nom du fichier encode le nom du PDF source, le numéro de page et la classe détectée.

In [22]:
print(pdf_path)

uploaded_images/Accuphase-E202amp.pdf


In [23]:
for p_id, page in tqdm(enumerate(pages), total=len(pages), desc="Traitement des pages"):
    page_array = np.array(page)

    # Prédiction YOLO page par page
   # result = model_extraction.predict(source=page, conf=0.212, verbose=False)
    result = model_extraction.predict(source=page, conf=0.212, verbose=False,device="cpu")

    for box in result[0].boxes:
        # Coordonnées de la zone détectée
        x, y, y1, x1 = map(int, box.xyxy[0])

        # Découpe avec marge
        margin = 10
        y1_m = max(0, y - margin)
        y2_m = min(page_array.shape[0], y1 + margin)
        x1_m = max(0, x - margin)
        x2_m = min(page_array.shape[1], x1 + margin)

        crop = page_array[y1_m:y2_m, x1_m:x2_m]

        if crop is not None and crop.size > 0:
            cls = result[0].names[int(box.cls[0])]
            
            pdf_path_str = next(iter(pdf_path))
            filename = f"{Path(pdf_path_str).stem}_{p_id}_{cls}.png"
            
            cv2.imwrite(os.path.join(dir_path, filename), crop)

print("Le traitement du fichier est terminé.")


NameError: name 'pages' is not defined

---
## 3. `test1.py` — Interface interactive de visualisation

Le but du script est de fournir une interface Matplotlib qui affiche côte à côte la table BOM, le schéma et la carte PCB, et permet de **cliquer sur un composant** (ex: `R12`) pour le localiser dans les trois images simultanément.

Il utilise deux moteurs de détection selon le type d'image :
- **Tesseract seul** pour la table BOM (texte structuré en colonnes)
- **YOLO + Tesseract** pour le schéma et la carte (étiquettes petites et dispersées)

**Usage standalone :**
```bash
python3 test1.py table.png schema.png carte.png
```

### 3.1 Imports, configuration et constantes

Le script tente de forcer le backend graphique `TkAgg` pour Matplotlib (nécessaire sur certains systèmes Linux/Mac pour afficher une fenêtre interactive). Le chargement du modèle `bestr.pt` est fait au niveau module — si le fichier est absent, le script s'arrête immédiatement.

`PREFIXES_VALIDES` liste les codes normalisés des composants électroniques : `R` = résistance, `C` = condensateur, `U` = circuit intégré, `D` = diode, `Q` = transistor, `L` = inductance, etc.

In [24]:
import cv2, re
import numpy as np
import pytesseract
from ultralytics import YOLO
import matplotlib
try:
    matplotlib.use('TkAgg')
except:
    pass
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors

PATH_TO_YOLO_MODEL = "models/best_table.pt"

try:
    yolo_model = YOLO(PATH_TO_YOLO_MODEL)
    print("Modèle YOLO chargé.")
except Exception as e:
    print(f"Erreur chargement YOLO : {e}")

PREFIXES_VALIDES = ['R', 'C', 'U', 'D', 'Q', 'L', 'J', 'SW', 'F', 'TP', 'RV', 'Y', 'DZ', 'IC']


Modèle YOLO chargé.


### 3.2 `natural_sort_key()` — tri naturel des références

Un tri alphabétique classique produirait `R1, R10, R11, R2` (car `'1' < '2'` mais `'10' < '2'` en comparaison de chaînes). Le tri naturel compare les parties numériques comme des entiers, donnant l'ordre correct : `R1, R2, R10, R11`.

La fonction découpe la chaîne en alternant segments texte et segments numériques, puis compare chaque segment dans son type propre.

In [25]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

### 3.3 `clean_ocr_text()` — correction des erreurs OCR

L'OCR fait souvent des erreurs visuelles sur les petites polices des schémas. Cette fonction applique un pipeline de corrections en 4 étapes :

1. **Nettoyage** : ne garder que les caractères alphanumériques majuscules (supprimer ponctuation, espaces, caractères parasites)
2. **Extraction du préfixe** : chercher un préfixe valide dans le texte. Les préfixes sont triés par longueur décroissante pour matcher `SW` avant `S`, `IC` avant `I`, etc.
3. **Correction des confusions visuelles** : `S→5`, `A→4`, `O→0`, `I→1`, `L→1`, `Z→2`, `G→6`, `B→8`
4. **Suppression des zéros de tête** : `Q010 → Q10`, `R001 → R1`

Si aucun préfixe valide n'est trouvé, la fonction retourne `None` pour indiquer que le texte ne correspond pas à une référence de composant.

In [26]:
def clean_ocr_text(text):
    """
    Nettoyage strict : 
    1. AR4 -> R4 (Suppression préfixes fantômes)
    2. Q010 -> Q10 (Suppression zéros de remplissage)
    3. RS -> R5 (Correction confusions visuelles)
    """
    t = re.sub(r'[^A-Z0-9]', '', text.upper())
    
    found_prefix = None
    remaining_part = ""
    
    for p in sorted(PREFIXES_VALIDES, key=len, reverse=True):
        match = re.search(r'(' + p + r')([A-Z0-9]+)', t)
        if match:
            found_prefix = match.group(1)
            remaining_part = match.group(2)
            break
    
    if found_prefix:
        suffix = remaining_part.replace('S', '5').replace('A', '4').replace('O', '0')
        suffix = suffix.replace('I', '1').replace('L', '1').replace('Z', '2')
        suffix = suffix.replace('G', '6').replace('B', '8')
        
        num_match = re.search(r'\d+', suffix)
        if num_match:
            num_str = num_match.group().lstrip('0')
            if not num_str: num_str = "0"
            return found_prefix + num_str
            
    return None

### 3.4 `preprocess_for_ocr()` — préparation d'une image pour Tesseract

Tesseract donne de bien meilleurs résultats sur des images :
- **en niveaux de gris** (pas de couleur à interpréter)
- **agrandies** (le texte des étiquettes est souvent très petit)
- **binarisées** (uniquement noir ou blanc, pas de nuances de gris)

La binarisation utilise la méthode d'Otsu : elle calcule automatiquement le seuil optimal qui sépare le fond du texte, sans paramètre à régler manuellement. Le paramètre `factor` contrôle le niveau d'agrandissement (3× par défaut, 4× pour les très petites étiquettes).

In [27]:
def preprocess_for_ocr(img, factor=3):
    """ Prépare une petite zone de l'image pour Tesseract """
    if img.size == 0: return img
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=factor, fy=factor, interpolation=cv2.INTER_CUBIC)
    return cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]

### 3.5 `detect_refs_tesseract()` — analyse de la table BOM

La table BOM est un tableau structuré où les références de composants se trouvent dans la **colonne de gauche** (approximativement les 18 premiers % de la largeur). On n'analyse que cette bande pour éviter de lire les valeurs, descriptions ou quantités des autres colonnes.

`image_to_data` (avec `output_type=DICT`) est préféré à `image_to_string` car il retourne aussi les **coordonnées** de chaque mot détecté — indispensable pour savoir où zoomer ensuite dans l'interface.

Le mode `--psm 11` demande à Tesseract de chercher des mots épars (sans supposer de mise en page particulière). La variable `last_prefix` permet de gérer les cellules qui listent juste des numéros (`12, 15, 18`) sans répéter le préfixe — on réutilise alors le dernier préfixe vu (`R12, R15, R18`).

In [28]:
def detect_refs_tesseract(img):
    """ Analyse de la table (BOM) - Colonne de gauche """
    if img is None: return []
    h, w = img.shape[:2]
    crop = img[:, :int(w*0.18)]
    proc = preprocess_for_ocr(crop, factor=2)
    data = pytesseract.image_to_data(proc, config='--oem 3 --psm 11', output_type=pytesseract.Output.DICT)
    
    found_refs = []
    last_prefix = "R"
    
    for i, txt in enumerate(data["text"]):
        clean = txt.strip().upper()
        if len(clean) < 2: continue
        
        parts = re.split(r",", clean)
        for p in parts:
            m = re.match(r"([A-Z]+)(\d+)", p)
            if m:
                last_prefix = m.group(1)
                found_refs.append({"ref": last_prefix + m.group(2), "cx": data["left"][i]/2, "cy": data["top"][i]/2, "bbox": (data["left"][i]/2, data["top"][i]/2, (data["left"][i]+data["width"][i])/2, (data["top"][i]+data["height"][i])/2)})
            else:
                num = re.search(r"\d+", p)
                if num:
                    found_refs.append({"ref": last_prefix + num.group(), "cx": data["left"][i]/2, "cy": data["top"][i]/2, "bbox": (data["left"][i]/2, data["top"][i]/2, (data["left"][i]+data["width"][i])/2, (data["top"][i]+data["height"][i])/2)})
    return found_refs

### 3.6 `detect_refs_hybrid()` — analyse du schéma et de la carte

Pour les schémas et cartes PCB, les étiquettes de composants sont petites, orientées dans tous les sens, et dispersées sur toute l'image. Une approche OCR globale produirait trop de bruit. On adopte donc une stratégie en deux temps :

1. **YOLO `bestr.pt`** localise les zones de l'image qui contiennent probablement une étiquette (`imgsz=1280` : taille d'entrée agrandie pour mieux détecter les très petits éléments)
2. Pour chaque zone détectée, **Tesseract** lit le texte en mode `--psm 7` (ligne de texte unique) avec une liste blanche de caractères limitée aux lettres et chiffres attendus
3. `clean_ocr_text()` valide et normalise le résultat — si la chaîne lue ne correspond à aucune référence valide, elle est ignorée

In [29]:
def detect_refs_hybrid(img):
    """ Analyse du Schéma/Carte : YOLO (où ?) + OCR (quoi ?) """
    if img is None: return []
    results = yolo_model.predict(source=img, conf=0.20, imgsz=1280, verbose=False)
    found_refs = []
    
    for result in results:
        for box in result.boxes:
            b = box.xyxy[0].cpu().numpy().astype(int)
            roi = img[max(0, b[1]-5):b[3]+5, max(0, b[0]-5):b[2]+5]
            if roi.size > 0:
                roi_proc = preprocess_for_ocr(roi, factor=4)
                config = '--psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
                raw_text = pytesseract.image_to_string(roi_proc, config=config).strip()
                
                final_ref = clean_ocr_text(raw_text)
                if final_ref:
                    found_refs.append({
                        "ref": final_ref, 
                        "cx": float((b[0] + b[2]) / 2), 
                        "cy": float((b[1] + b[3]) / 2), 
                        "bbox": (float(b[0]), float(b[1]), float(b[2]), float(b[3]))
                    })
    return found_refs

### 3.7 `afficher()` — construction de l'interface Matplotlib

C'est le point d'entrée de l'interface. Elle charge les images, lance les détections sur chacune, puis construit la figure Matplotlib avec 3 ou 4 panneaux selon si la carte est fournie.

Les résultats des détections sont stockés dans des dictionnaires `{référence: infos}` (ex: `{"R12": {"cx": 450, "cy": 230, "bbox": (...)}}`). Cela permet une recherche en O(1) lors des interactions.

In [30]:
def afficher(table_path, schema_path, carte_path=None):
    img_t = cv2.imread(table_path)
    img_s = cv2.imread(schema_path)
    img_c = cv2.imread(carte_path) if carte_path else None

    print("Analyse en cours...")
    dict_table  = {r["ref"]: r for r in detect_refs_tesseract(img_t)}
    dict_schema = {r["ref"]: r for r in detect_refs_hybrid(img_s)}
    dict_carte  = {r["ref"]: r for r in detect_refs_hybrid(img_c)} if img_c is not None else {}

    print(f" Bilan : Table({len(dict_table)}) | Schéma({len(dict_schema)}) | Carte({len(dict_carte)})")

    fig = plt.figure(figsize=(18, 9))
    cols = 4 if img_c is not None else 3
    gs = fig.add_gridspec(1, cols, width_ratios=[1.2, 2, 2, 2][:cols])

    ax_list = fig.add_subplot(gs[0,0])
    ax_table = fig.add_subplot(gs[0,1])
    ax_schema = fig.add_subplot(gs[0,2])
    ax_carte = fig.add_subplot(gs[0,3]) if img_c is not None else None
    axes_images = [ax for ax in [ax_table, ax_schema, ax_carte] if ax]

    ax_table.imshow(cv2.cvtColor(img_t, cv2.COLOR_BGR2RGB))
    ax_schema.imshow(cv2.cvtColor(img_s, cv2.COLOR_BGR2RGB))
    if ax_carte: ax_carte.imshow(cv2.cvtColor(img_c, cv2.COLOR_BGR2RGB))
    for ax in [ax_list] + axes_images: ax.axis("off")
    ids = sorted(dict_table.keys(), key=natural_sort_key)
    colors = list(mcolors.TABLEAU_COLORS.values())
    text_mapping, y_pos, last_prefix, color_idx = {}, 0.98, None, 0
    
    for ident in ids:
        m = re.match(r"([A-Z]+)", ident)
        prefix = m.group(1) if m else "?"
        if prefix != last_prefix:
            if last_prefix: y_pos -= 0.02
            ax_list.text(0.05, y_pos, f"--- {prefix} ---", fontsize=10, fontweight='bold', alpha=0.7)
            y_pos -= 0.03
            color_idx = (color_idx + 1) % len(colors)
        
        txt_obj = ax_list.text(0.15, y_pos, ident, fontsize=9, picker=5, fontweight='bold', color=colors[color_idx])
        text_mapping[txt_obj] = ident
        last_prefix, y_pos = prefix, y_pos - 0.025
        if y_pos < 0.02: break
                # Rectangles de focus
    rects = {ax: patches.Rectangle((0,0),0,0, edgecolor="red", facecolor="none", linewidth=2, visible=False) for ax in axes_images}
    for ax in axes_images: ax.add_patch(rects[ax])
    orig_lims = {ax: (ax.get_xlim(), ax.get_ylim()) for ax in axes_images}

    def do_focus(ident):
        zoom = 350
        found = False
        for ax, dic in [(ax_table, dict_table), (ax_schema, dict_schema), (ax_carte, dict_carte)]:
            if ax and ident in dic:
                info = dic[ident]
                ax.set_xlim(info["cx"] - zoom, info["cx"] + zoom)
                ax.set_ylim(info["cy"] + zoom, info["cy"] - zoom)
                b = info["bbox"]
                rects[ax].set_xy((b[0], b[1]))
                rects[ax].set_width(b[2]-b[0]); rects[ax].set_height(b[3]-b[1])
                rects[ax].set_visible(True)
                found = True
            elif ax:
                rects[ax].set_visible(False)
        if found:
            fig.suptitle(f"Focus : {ident}", color="red", fontsize=16, fontweight="bold")
            plt.draw()

    def on_click(event):
        if event.button == 3: # Reset
            for ax in axes_images:
                ax.set_xlim(orig_lims[ax][0]); ax.set_ylim(orig_lims[ax][1])
                rects[ax].set_visible(False)
            fig.suptitle("")
            plt.draw()
        elif event.inaxes in axes_images and event.button == 1:
            curr_dict = {ax_table: dict_table, ax_schema: dict_schema, ax_carte: dict_carte}.get(event.inaxes)
            if curr_dict:
                pts = np.array([[r["cx"], r["cy"]] for r in curr_dict.values()])
                if len(pts) > 0:
                    dist = np.linalg.norm(pts - [event.xdata, event.ydata], axis=1)
                    idx = np.argmin(dist)
                    if dist[idx] < 200: do_focus(list(curr_dict.keys())[idx])

    fig.canvas.mpl_connect("pick_event", lambda e: do_focus(text_mapping[e.artist]))
    fig.canvas.mpl_connect("button_press_event", on_click)
    fig.canvas.mpl_connect("scroll_event", lambda e: on_scroll(e, axes_images))
    
    print("Interface prête.")
    plt.tight_layout()
    plt.show()    



def on_scroll(event, axes_images):
    if event.inaxes not in axes_images: return
    ax = event.inaxes
    scale = 1/1.5 if event.button == 'up' else 1.5
    cur_xlim, cur_ylim = ax.get_xlim(), ax.get_ylim()
    new_w, new_h = (cur_xlim[1]-cur_xlim[0])*scale, (cur_ylim[1]-cur_ylim[0])*scale
    ax.set_xlim([event.xdata - new_w/2, event.xdata + new_w/2])
    ax.set_ylim([event.ydata + new_h/2, event.ydata - new_h/2])
    plt.draw()


### 3.8 Panneau de liste cliquable

Les composants de la table BOM sont affichés sous forme de texte Matplotlib dans le panneau de gauche. Le paramètre `picker=5` rend chaque texte cliquable (zone de 5px autour du texte). Les composants sont groupés par préfixe et colorés par groupe (10 couleurs distinctes issues de `TABLEAU_COLORS`).

`text_mapping` associe chaque objet texte Matplotlib à son identifiant de composant — indispensable pour savoir sur quel composant l'utilisateur a cliqué dans le callback `pick_event`.

### 3.9 Rectangles de focus et fonction `do_focus()`

Un rectangle `patches.Rectangle` invisible est pré-créé pour chaque panneau image. Quand `do_focus()` est appelée avec un identifiant de composant, elle :
1. Cherche le composant dans chaque dictionnaire
2. Si trouvé : zoome le panneau autour de la position du composant (`set_xlim` / `set_ylim`), repositionne et rend visible le rectangle rouge
3. Si absent de ce panneau : masque le rectangle

Note : en Matplotlib, l'axe Y est inversé par rapport aux coordonnées image (origine en haut), d'où `set_ylim(cy + zoom, cy - zoom)` pour zoomer correctement.

### 3.10 Gestion des événements souris

Trois types d'interactions sont disponibles dans l'interface :

| Interaction | Effet |
|---|---|
| Clic gauche sur la liste | Zoom sur le composant dans les 3 images + rectangle rouge |
| Clic gauche sur une image | Trouve le composant le plus proche du clic (distance euclidienne < 200px) |
| Clic droit n'importe où | Réinitialise toutes les vues à leur niveau de zoom original |
| Molette sur une image | Zoom libre centré sur la position du curseur (×1.5 par cran) |

La recherche du composant le plus proche utilise `np.linalg.norm` pour calculer la distance euclidienne entre le point cliqué et tous les centres de composants connus, puis `np.argmin` pour trouver le plus proche.

### 3.11 `on_scroll()` — zoom à la molette

Le zoom est implémenté en redimensionnant les limites de l'axe autour du point pointé par le curseur. Un facteur 1.5 est appliqué à chaque cran de molette. Scroll vers le haut = zoom avant (division par 1.5), scroll vers le bas = zoom arrière (multiplication par 1.5).

---
## 2. `merge.py` — Orchestration de la lecture des images

Ce script est une version améliorée de `prediction.py`. Au lieu de sauvegarder toutes les détections, il ne conserve que **la meilleure zone par classe** (celle avec la plus grande surface en pixels), applique des rotations si nécessaire, puis lance l'interface interactive de `test1.py`.

**Usage :**
```bash
python3 merge.py Accuphase-E202amp.pdf ../../../models/extraction/best.pt
```

### 2.1 Imports et import de l'interface

### 2.2 Fonction principale `process_and_visualize()`

Toute la logique est encapsulée dans cette fonction. Elle reçoit le chemin du PDF et le chemin du modèle YOLO, et orchestre les 4 étapes suivantes.

#### Étape 1 & 2 : Chargement du modèle et conversion du PDF

Identique à `prediction.py` : le modèle YOLO est chargé, puis le PDF est converti page par page en images PIL à 300 DPI.

In [31]:
import os
import sys
import cv2
import numpy as np
from pdf2image import convert_from_path
from ultralytics import YOLO

# afficher() doit déjà être définie (section test1.py ci-dessus)


#### Étape 2 : Sélection de la meilleure zone par classe

Le dictionnaire `best_images` stocke pour chaque classe un tuple `(image_découpée, surface_en_pixels)`. À chaque nouvelle détection, on calcule la surface (`largeur × hauteur`) et on ne garde la découpe que si elle est plus grande que celle déjà enregistrée. Ainsi, à la fin du parcours de toutes les pages, on a retenu la zone la plus grande pour chaque type.

In [32]:
def process_and_visualize(pdf_path, model_path):
    # 1. Chargement du modèle de zones (prediction.py)
    try:
        model = YOLO(model_path)
    except Exception as e:
        print(f"Erreur chargement modèle : {e}")
        return

    # 2. Conversion PDF en images (DPI 300)
    print("--- Conversion du PDF en images ---")
    pages = convert_from_path(pdf_path, dpi=300)
    
    best_images = {"Table": (None, 0), "Diagram": (None, 0), "Board": (None, 0)}

    print("--- Extraction des zones via YOLO ---")
    for p_id, page in enumerate(pages):
        # Utilisation directe du format PIL pour la détection comme dans prediction.py
        results = model.predict(source=page, conf=0.212, verbose=False)
        page_array = np.array(page)

        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]
            
            # Mapping vers nos clés
            key = None
            if cls_name.lower() == "table": key = "Table"
            elif cls_name.lower() == "diagram": key = "Diagram"
            elif cls_name.lower() == "board": key = "Board"

            if key:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                # Découpage avec marge de 10px
                margin = 10
                y1_m, y2_m = max(0, y1 - margin), min(page_array.shape[0], y2 + margin)
                x1_m, x2_m = max(0, x1 - margin), min(page_array.shape[1], x2 + margin)
                
                crop = page_array[y1_m:y2_m, x1_m:x2_m]
                
                if crop is not None and crop.size > 0:
                    surface = crop.shape[0] * crop.shape[1]
                    if surface > best_images[key][1]:
                        best_images[key] = (crop, surface)

    # 3. Traitement, Rotations et Sauvegarde
    temp_paths = {}
    for cls in ["Table", "Diagram", "Board"]:
        img_data, surf = best_images[cls]
        if img_data is not None:
            # Conversion RGB vers BGR pour OpenCV
            img_to_save = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)

            # --- APPLICATION DES ROTATIONS ---
            if cls == "Diagram":
                # Rotation 90° vers la GAUCHE (Counter-Clockwise)
                print("Rotation du Schéma (90° gauche)...")
                img_to_save = cv2.rotate(img_to_save, cv2.ROTATE_90_COUNTERCLOCKWISE)
            
            elif cls == "Board":
                # Rotation 90° vers la DROITE (Clockwise)
                print("Rotation de la Carte (90° droite)...")
                img_to_save = cv2.rotate(img_to_save, cv2.ROTATE_90_CLOCKWISE)

            path = f"temp_{cls.lower()}.png"
            cv2.imwrite(path, img_to_save)
            temp_paths[cls] = path
 
    if "Table" in temp_paths and "Diagram" in temp_paths:
        print(f"--- Affichage ---")
        afficher(
            temp_paths["Table"], 
            temp_paths["Diagram"], 
            temp_paths.get("Board")
        )
    else:
        print("\nERREUR : 'Table' ou 'Diagram' manquant.")
        print(f"Trouvés : {[k for k, v in best_images.items() if v[0] is not None]}")
        

#### Étape 3 : Rotations et sauvegarde temporaire

Les schémas et cartes sont souvent orientés en mode paysage dans les PDFs. On les redresse avant affichage :

- `Diagram` → rotation 90° **gauche** (counter-clockwise)
- `Board` → rotation 90° **droite** (clockwise)  
- `Table` → aucune rotation

On effectue aussi la conversion de couleurs **RGB → BGR**, nécessaire car PIL produit des images en RGB tandis qu'OpenCV (`cv2.imwrite`) attend du BGR.

#### Étape 4 : Lancement de l'interface

Table et Diagram sont obligatoires pour que l'interface ait du sens. Board est optionnel — .get() retourne None sans lever d'erreur si la clé est absente, et afficher() gère ce cas.

On utilise process_and_visualize_v2() qui integre l'OCR apres le merge.

---
## OCR des étiquettes composants — `extract_easyocr.py` 

Cette section s'insère **après le merge** (étape 2) et **avant l'interface** (étape 3).  
Une fois que `process_and_visualize()` a sélectionné les meilleures images (Diagram, Board),  
le pipeline OCR est appliqué pour lire les étiquettes des composants avec une précision maximale.

**Pipeline OCR v7 :**
1. Auto-détection des paramètres selon la résolution (DPI estimé, contraste, fond)
2. Pré-upscale si résolution faible (< 150 DPI estimé)
3. YOLO localise les zones d'étiquettes
4. **Merge des boîtes voisines** — fusionne `"R"` + `"516"` en `"R516"` avant OCR
5. EasyOCR sur chaque crop (3 tentatives : CLAHE → CLAHE+rot90 → Otsu)
6. Correcteur OCR scan linéaire (confusions `I↔1`, `O↔0`, `G↔6`…)
7. Dictionnaire de post-corrections (erreurs systématiques connues)
8. Déduplication — même ref détectée N fois → on garde le meilleur score


In [33]:
import time
import easyocr

print('\u23f3  Chargement EasyOCR (langues latines)…', end='', flush=True)
_t0 = time.time()
# Langues a script latin - partagent le meme modele sous-jacent, pas de surcout memoire.
# Ameliore la robustesse sur les polices techniques variees des schemas electroniques.
READER = easyocr.Reader(
    ['en', 'fr', 'de', 'es', 'it', 'nl', 'pt'],
    gpu=True, verbose=False
)
print(f'  ✔  EasyOCR pret  ({time.time()-_t0:.1f}s)')


⏳  Chargement EasyOCR (langues latines)…  ✔  EasyOCR pret  (2.2s)


### Correcteur OCR, merge de boites, preprocessing et pipeline

In [34]:
import re as _re

# Constantes
ALLOWLIST    = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-_'
OCR_CONF_MIN = 0.15

# Correcteur de confusions OCR
PREFIX_FIXES = {'0': 'O', '1': 'I', '8': 'B', '6': 'G', '5': 'S'}
DIGIT_FIXES  = {
    'I': '1', 'J': '1', 'L': '1', '|': '1',
    'O': '0', 'Q': '0', 'D': '0',
    'G': '6', 'Z': '2', 'S': '5', 'B': '8', 'T': '7',
}

def fix_ocr_confusion(text: str) -> str:
    # Scan lineaire : trouve le 1er vrai chiffre comme frontiere prefixe/numerique.
    # Avant : corrige chiffres mal lus en lettres
    # Apres : corrige lettres mal lues en chiffres
    # Ex : 'C41J'->'C411', 'R51G'->'R516', '0C402'->'OC402', 'RI2'->'R12'
    if not text:
        return text
    text = text.strip().upper().replace(' ', '')
    i = 0
    while i < len(text) and not text[i].isdigit():
        i += 1
    if i == len(text):
        return text

    prefix_raw = text[:i]
    rest       = text[i:]

    VALID_2L = {'IC','TR','VR','BC','ZD','SW','TP','FR','ST','CF',
                'FL','RL','DL','LED','SCR','FET','MOV'}
    if (len(prefix_raw) >= 2 and prefix_raw[-1] == 'I'
            and prefix_raw not in VALID_2L
            and prefix_raw[:-1] + prefix_raw[-1] not in VALID_2L):
        prefix_raw = prefix_raw[:-1]
        rest = '1' + rest

    if len(rest) > 1 and rest[-1].isalpha() and rest[-1] not in DIGIT_FIXES:
        suffix, digits_raw = rest[-1], rest[:-1]
    else:
        suffix, digits_raw = '', rest

    fixed_prefix = ''.join(
        PREFIX_FIXES.get(ch, ch) if ch.isdigit() else ch for ch in prefix_raw)
    fixed_digits = ''.join(
        ch if ch.isdigit() else DIGIT_FIXES.get(ch, ch) for ch in digits_raw)
    return fixed_prefix + fixed_digits + suffix


# Post-corrections (dictionnaire des erreurs systematiques connues)
POST_CORRECTIONS = {
    'UC402': 'IC402', 'UC405': 'IC405', 'RSI9': 'R519', 'RL26': 'R126',
    'S501': 'C501', 'B178': 'BC178', 'BCI78': 'BC178', 'BCI7': 'BC17',
    'CI7A': 'C17A', 'LC41': 'C41', 'ECN8': 'C8', 'IB6': 'B6',
    'REO7': 'R07', 'JEC740': 'C740', 'STLO1': 'ST01',
    'R62I': 'R621', 'C41J': 'C411', 'C50G': 'C506', 'R5122': 'R512',
    'BC1788': 'BC178', 'BC178B': 'BC178', 'T5502': 'T502',
}

def apply_post_corrections(text: str) -> str:
    key = text.strip().upper().replace(' ', '')
    return POST_CORRECTIONS.get(key, text)


# Validation regex
COMPONENT_PATTERN = _re.compile(
    r'^('
    r'[A-Z]{1,2}\s?\d{3,5}[A-Z]?'
    r'|[A-Z]{1,3}\d{2,5}[A-Z]?'
    r'|[A-Z]{2,4}\d{1,4}[A-Z]?'
    r'|St\s?\d{3}'
    r'|[A-Z]{1,3}\d+[-_]\d+'
    r')$',
    _re.IGNORECASE
)

def is_valid_ref(text: str) -> bool:
    return bool(COMPONENT_PATTERN.match(text.strip()))


# Merge des boites voisines
def merge_nearby_boxes(boxes, gap_ratio=0.8, overlap_y_ratio=0.4):
    # Fusionne les paires de boites probablement issues du meme label.
    # YOLO peut separer 'R' et '516' en 2 boites -> fusionne en 'R516'.
    if not boxes:
        return boxes
    boxes = sorted(boxes, key=lambda b: b[0])
    merged = list(boxes)
    changed = True
    while changed:
        changed = False
        new_merged, used = [], [False] * len(merged)
        for i in range(len(merged)):
            if used[i]: continue
            x1a, y1a, x2a, y2a, ca = merged[i]
            best_j, best_gap = -1, float('inf')
            for j in range(i + 1, len(merged)):
                if used[j]: continue
                x1b, y1b, x2b, y2b, cb = merged[j]
                if x1b < x2a: continue
                h_avg = ((y2a - y1a) + (y2b - y1b)) / 2
                gap   = x1b - x2a
                if gap > gap_ratio * h_avg: continue
                overlap_y = min(y2a, y2b) - max(y1a, y1b)
                min_h = min(y2a - y1a, y2b - y1b)
                if min_h == 0 or overlap_y / min_h < overlap_y_ratio: continue
                if gap < best_gap:
                    best_gap, best_j = gap, j
            if best_j >= 0:
                x1b, y1b, x2b, y2b, cb = merged[best_j]
                new_merged.append((min(x1a,x1b), min(y1a,y1b),
                                   max(x2a,x2b), max(y2a,y2b), (ca+cb)/2))
                used[i] = used[best_j] = True
                changed = True
            else:
                new_merged.append(merged[i]); used[i] = True
        merged = new_merged
    return merged


# Pretraitements crop
def preprocess_crop(crop_bgr, scale=5):
    h, w = crop_bgr.shape[:2]
    big  = cv2.resize(crop_bgr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    k    = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    sharp = cv2.filter2D(big, -1, k)
    gray  = cv2.cvtColor(sharp, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    bgr = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    return cv2.copyMakeBorder(bgr, 10, 10, 10, 10, cv2.BORDER_CONSTANT, value=(255, 255, 255))


def preprocess_crop_otsu(crop_bgr, scale=5):
    h, w = crop_bgr.shape[:2]
    big  = cv2.resize(crop_bgr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bgr = cv2.cvtColor(bw, cv2.COLOR_GRAY2BGR)
    return cv2.copyMakeBorder(bgr, 10, 10, 10, 10, cv2.BORDER_CONSTANT, value=(255, 255, 255))


# EasyOCR sur un crop
def _run_easyocr(img_bgr):
    try:
        results = READER.readtext(
            img_bgr, detail=1, paragraph=False,
            allowlist=ALLOWLIST, width_ths=0.7,
        )
    except Exception:
        return []
    out = []
    for (_, text, conf) in results:
        if conf < OCR_CONF_MIN: continue
        t = _re.sub(r'\s+', '', text.strip().upper())
        if t: out.append((t, conf))
    return sorted(out, key=lambda x: x[1], reverse=True)


def run_ocr(crop_bgr, scale=5):
    # 3 tentatives successives, arret des qu'une ref valide est trouvee :
    # 1. CLAHE normal
    # 2. CLAHE + rotation 90 (crop vertical ou rien trouve)
    # 3. Otsu (fallback fond sombre / contraste inverse)
    # Retourne (text_corrige, is_valid, conf).
    def best_from(img):
        hits = _run_easyocr(img)
        if not hits: return '', False, 0.0
        for (t, c) in hits:
            fixed = apply_post_corrections(fix_ocr_confusion(t))
            if is_valid_ref(fixed): return fixed, True, c
        fixed = apply_post_corrections(fix_ocr_confusion(hits[0][0]))
        return fixed, False, hits[0][1]

    pre_clahe = preprocess_crop(crop_bgr, scale=scale)
    h, w = pre_clahe.shape[:2]
    text, valid, conf = best_from(pre_clahe)
    if valid: return text, valid, conf

    if h > w * 1.3 or not text:
        t2, v2, c2 = best_from(cv2.rotate(pre_clahe, cv2.ROTATE_90_CLOCKWISE))
        if v2 or (t2 and c2 > conf):
            text, valid, conf = t2, v2, c2
    if valid: return text, valid, conf

    pre_otsu = preprocess_crop_otsu(crop_bgr, scale=scale)
    t3, v3, c3 = best_from(pre_otsu)
    if v3 or (t3 and c3 > conf): return t3, v3, c3
    return text, valid, conf


# Deduplication des refs identiques
def deduplicate_refs(detections):
    # Si la meme ref valide apparait N fois, on garde le meilleur score.
    seen, result = {}, []
    for d in detections:
        if not d['is_valid_ref']: result.append(d); continue
        key   = d['text'].replace(' ', '').upper()
        score = d['yolo_conf'] * d['ocr_conf']
        if key not in seen:
            seen[key] = len(result); result.append(d)
        else:
            ex = result[seen[key]]
            if score > ex['yolo_conf'] * ex['ocr_conf']:
                result[seen[key]] = d
    return result


# Auto-detection des parametres selon l'image
def auto_params(img):
    # Adapte imgsz YOLO, scale crop, conf et pre-upscale selon la resolution.
    h, w = img.shape[:2]
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mean  = float(gray.mean())
    std   = float(gray.std())
    dpi_est = int(w / (297 / 25.4))

    if   dpi_est < 80:  pre_upscale = 3
    elif dpi_est < 150: pre_upscale = 2
    else:               pre_upscale = 1

    eff_px = (w * pre_upscale) * (h * pre_upscale)
    if   eff_px > 3_000_000: imgsz, scale = 1280, 5
    elif eff_px > 1_000_000: imgsz, scale = 1024, 6
    elif eff_px > 300_000:   imgsz, scale = 640,  8
    else:                    imgsz, scale = 640,  10

    if   std < 25: conf = 0.25
    elif std < 45: conf = 0.30
    else:          conf = 0.40

    return {'imgsz': imgsz, 'scale': scale, 'conf': conf,
            'invert': mean < 80, 'pre_upscale': pre_upscale, 'dpi_est': dpi_est}


def upscale_image(img, factor):
    h, w = img.shape[:2]
    big  = cv2.resize(img, (w * factor, h * factor), interpolation=cv2.INTER_CUBIC)
    k    = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    return cv2.filter2D(big, -1, k)


print('✔  Fonctions OCR v7 definies.')


✔  Fonctions OCR v7 definies.


### Fonction principale : `run_ocr_on_image()`

In [35]:
import csv
import json
from datetime import datetime
from pathlib import Path

def run_ocr_on_image(img_path, yolo_model, output_dir='results', label=None):
    """
    Pipeline OCR v7 complet sur une image (Diagram ou Board).

    Parametres
    ----------
    img_path   : str  - chemin vers l'image produite par process_and_visualize
    yolo_model : YOLO - modele pour detecter les etiquettes
    output_dir : str  - dossier de sortie CSV / JSON / image annotee
    label      : str  - nom court pour les fichiers (ex: 'diagram')

    Retourne
    --------
    dict : {ref: {'cx','cy','bbox'}} compatible avec afficher()
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    img = cv2.imread(str(img_path))
    if img is None:
        print(f'⚠  Image introuvable : {img_path}')
        return {}

    stem = label or Path(img_path).stem
    print(f'\n{chr(9552)*55}')
    print(f'  OCR v7 -- {stem}  ({img.shape[1]}x{img.shape[0]} px)')
    print(f'{chr(9552)*55}')

    params = auto_params(img)
    print(f'  ~{params["dpi_est"]} DPI | pre_upscale x{params["pre_upscale"]}'
          f' | imgsz={params["imgsz"]} | scale={params["scale"]} | conf={params["conf"]}')

    if params['invert']:
        img = cv2.bitwise_not(img)
        print('  ⚠  Fond sombre -> image inversee')
    if params['pre_upscale'] > 1:
        img = upscale_image(img, params['pre_upscale'])
        print(f'  Pre-upscale x{params["pre_upscale"]} -> {img.shape[1]}x{img.shape[0]} px')

    # YOLO + merge boites
    t0 = time.time()
    yolo_res  = yolo_model(img, conf=params['conf'],
                           imgsz=params['imgsz'], verbose=False)[0]
    boxes_raw = [
        (int(b.xyxy[0][0]), int(b.xyxy[0][1]),
         int(b.xyxy[0][2]), int(b.xyxy[0][3]), float(b.conf[0]))
        for b in yolo_res.boxes
    ]
    boxes   = merge_nearby_boxes(boxes_raw)
    fusions = len(boxes_raw) - len(boxes)
    print(f'  YOLO : {len(boxes_raw)} zones -> {len(boxes)} apres merge '
          f'({fusions} fusions)  [{time.time()-t0:.1f}s]')

    # OCR crop par crop
    detections, PAD = [], 4
    for i, (x1, y1, x2, y2, conf_yolo) in enumerate(boxes):
        crop = img[max(0, y1-PAD):min(img.shape[0], y2+PAD),
                   max(0, x1-PAD):min(img.shape[1], x2+PAD)]
        if crop.size == 0: continue
        text, valid, ocr_conf = run_ocr(crop, scale=params['scale'])
        detections.append({
            'id':           i + 1,
            'text':         text,
            'is_valid_ref': valid,
            'yolo_conf':    round(conf_yolo, 3),
            'ocr_conf':     round(ocr_conf, 3),
            'score':        round(conf_yolo * ocr_conf, 3),
            'bbox':         {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2},
            'center':       {'x': (x1+x2)//2, 'y': (y1+y2)//2},
        })
        if (i+1) % 20 == 0 or i+1 == len(boxes):
            pct = int(100*(i+1)/len(boxes))
            print(f'\r  OCR : {i+1}/{len(boxes)} ({pct}%)', end='', flush=True)
    print()

    # Deduplication
    before     = sum(1 for d in detections if d['is_valid_ref'])
    detections = deduplicate_refs(detections)
    after      = sum(1 for d in detections if d['is_valid_ref'])
    valid_dets = [d for d in detections if d['is_valid_ref']]
    high_conf  = [d for d in valid_dets if d['ocr_conf'] >= 0.7]
    print(f'  ✔  Refs valides : {after}  (dedup {before}->{after})'
          f'  |  haute conf (>=0.7) : {len(high_conf)}')

    # Exports
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Image annotee
    ann = img.copy()
    for d in detections:
        b = d['bbox']
        color = (0, 200, 0) if d['is_valid_ref'] else (0, 140, 255)
        cv2.rectangle(ann, (b['x1'],b['y1']), (b['x2'],b['y2']), color, 2)
        if d['text']:
            fs = 0.4
            (tw, th), _ = cv2.getTextSize(d['text'], cv2.FONT_HERSHEY_SIMPLEX, fs, 1)
            cv2.rectangle(ann, (b['x1'], b['y1']-th-6),
                          (b['x1']+tw+4, b['y1']), color, -1)
            cv2.putText(ann, d['text'], (b['x1']+2, b['y1']-3),
                        cv2.FONT_HERSHEY_SIMPLEX, fs, (255,255,255), 1)
    ann_path = os.path.join(output_dir, f'{stem}_{ts}_annotated.png')
    cv2.imwrite(ann_path, ann)

    # CSV
    csv_path = os.path.join(output_dir, f'{stem}_{ts}_ocr.csv')
    fields   = ['id','text','is_valid_ref','yolo_conf','ocr_conf','score',
                'x1','y1','x2','y2','cx','cy']
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for d in detections:
            w.writerow({
                'id': d['id'], 'text': d['text'],
                'is_valid_ref': d['is_valid_ref'],
                'yolo_conf': d['yolo_conf'], 'ocr_conf': d['ocr_conf'],
                'score': d['score'],
                'x1': d['bbox']['x1'], 'y1': d['bbox']['y1'],
                'x2': d['bbox']['x2'], 'y2': d['bbox']['y2'],
                'cx': d['center']['x'], 'cy': d['center']['y'],
            })

    # JSON
    json_path = os.path.join(output_dir, f'{stem}_{ts}_ocr.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(detections, f, ensure_ascii=False, indent=2)

    print(f'  -> {ann_path}')
    print(f'  -> {csv_path}')
    print(f'  -> {json_path}')

    # Retourne le dict compatible afficher()
    return {
        d['text']: {
            'ref':  d['text'],
            'cx':   d['center']['x'],
            'cy':   d['center']['y'],
            'bbox': (d['bbox']['x1'], d['bbox']['y1'],
                     d['bbox']['x2'], d['bbox']['y2']),
        }
        for d in valid_dets
    }

print('run_ocr_on_image() definie.')


run_ocr_on_image() definie.


### `afficher()` — version mise a jour (support OCR)

La signature est etendue avec deux parametres optionnels `ocr_schema` et `ocr_carte`.
Si fournis, ils remplacent `detect_refs_hybrid()` par les resultats OCR (plus precis).
Sans ces parametres, le comportement est **identique a l'original** (fallback Tesseract).


In [36]:
def afficher(table_path, schema_path, carte_path=None,
             ocr_schema=None, ocr_carte=None):
    """
    Interface interactive Matplotlib.

    Parametres
    ----------
    table_path, schema_path, carte_path : chemins des images
    ocr_schema : dict optionnel {ref: {cx,cy,bbox}} issu de run_ocr_on_image() [OCR v7]
    ocr_carte  : dict optionnel {ref: {cx,cy,bbox}} issu de run_ocr_on_image() [OCR v7]

    Si ocr_schema/ocr_carte sont None -> fallback detect_refs_hybrid() (comportement original).
    """
    import matplotlib.patches as patches
    import matplotlib.colors as mcolors

    img_t = cv2.imread(table_path)
    img_s = cv2.imread(schema_path)
    img_c = cv2.imread(carte_path) if carte_path else None

    print('Analyse en cours...')
    dict_table = {r['ref']: r for r in detect_refs_tesseract(img_t)}

    if ocr_schema is not None:
        dict_schema = ocr_schema
        print(f' Schema : {len(dict_schema)} refs (OCR v7)')
    else:
        dict_schema = {r['ref']: r for r in detect_refs_hybrid(img_s)}
        print(f' Schema : {len(dict_schema)} refs (YOLO+Tesseract)')

    if ocr_carte is not None:
        dict_carte = ocr_carte
        print(f' Carte  : {len(dict_carte)} refs (OCR v7)')
    elif img_c is not None:
        dict_carte = {r['ref']: r for r in detect_refs_hybrid(img_c)}
        print(f' Carte  : {len(dict_carte)} refs (YOLO+Tesseract)')
    else:
        dict_carte = {}

    print(f' Bilan : Table({len(dict_table)}) | Schema({len(dict_schema)}) | Carte({len(dict_carte)})')

    fig = plt.figure(figsize=(18, 9))
    cols = 4 if img_c is not None else 3
    gs = fig.add_gridspec(1, cols, width_ratios=[1.2, 2, 2, 2][:cols])

    ax_list   = fig.add_subplot(gs[0, 0])
    ax_table  = fig.add_subplot(gs[0, 1])
    ax_schema = fig.add_subplot(gs[0, 2])
    ax_carte  = fig.add_subplot(gs[0, 3]) if img_c is not None else None
    axes_images = [ax for ax in [ax_table, ax_schema, ax_carte] if ax]

    ax_table.imshow(cv2.cvtColor(img_t, cv2.COLOR_BGR2RGB))
    ax_schema.imshow(cv2.cvtColor(img_s, cv2.COLOR_BGR2RGB))
    if ax_carte: ax_carte.imshow(cv2.cvtColor(img_c, cv2.COLOR_BGR2RGB))
    for ax in [ax_list] + axes_images: ax.axis('off')

    ids = sorted(dict_table.keys(), key=natural_sort_key)
    colors = list(mcolors.TABLEAU_COLORS.values())
    text_mapping, y_pos, last_prefix, color_idx = {}, 0.98, None, 0

    for ident in ids:
        m = re.match(r'([A-Z]+)', ident)
        prefix = m.group(1) if m else '?'
        if prefix != last_prefix:
            if last_prefix: y_pos -= 0.02
            ax_list.text(0.05, y_pos, f'--- {prefix} ---',
                         fontsize=10, fontweight='bold', alpha=0.7)
            y_pos -= 0.03
            color_idx = (color_idx + 1) % len(colors)
        txt_obj = ax_list.text(0.15, y_pos, ident, fontsize=9, picker=5,
                               fontweight='bold', color=colors[color_idx])
        text_mapping[txt_obj] = ident
        last_prefix, y_pos = prefix, y_pos - 0.025
        if y_pos < 0.02: break

    rects = {ax: patches.Rectangle((0,0), 0, 0, edgecolor='red',
                                    facecolor='none', linewidth=2, visible=False)
             for ax in axes_images}
    for ax in axes_images: ax.add_patch(rects[ax])
    orig_lims = {ax: (ax.get_xlim(), ax.get_ylim()) for ax in axes_images}

    def do_focus(ident):
        zoom  = 350
        found = False
        for ax, dic in [(ax_table, dict_table), (ax_schema, dict_schema),
                        (ax_carte, dict_carte)]:
            if ax and ident in dic:
                info = dic[ident]
                ax.set_xlim(info['cx'] - zoom, info['cx'] + zoom)
                ax.set_ylim(info['cy'] + zoom, info['cy'] - zoom)
                b = info['bbox']
                rects[ax].set_xy((b[0], b[1]))
                rects[ax].set_width(b[2] - b[0]); rects[ax].set_height(b[3] - b[1])
                rects[ax].set_visible(True); found = True
            elif ax:
                rects[ax].set_visible(False)
        if found:
            fig.suptitle(f'Focus : {ident}', color='red', fontsize=16, fontweight='bold')
            plt.draw()

    def on_click(event):
        if event.button == 3:
            for ax in axes_images:
                ax.set_xlim(orig_lims[ax][0]); ax.set_ylim(orig_lims[ax][1])
                rects[ax].set_visible(False)
            fig.suptitle('')
            plt.draw()
        elif event.inaxes in axes_images and event.button == 1:
            curr_dict = {ax_table: dict_table, ax_schema: dict_schema,
                         ax_carte: dict_carte}.get(event.inaxes)
            if curr_dict:
                pts = np.array([[r['cx'], r['cy']] for r in curr_dict.values()])
                if len(pts) > 0:
                    dist = np.linalg.norm(pts - [event.xdata, event.ydata], axis=1)
                    idx  = np.argmin(dist)
                    if dist[idx] < 200: do_focus(list(curr_dict.keys())[idx])

    fig.canvas.mpl_connect('pick_event',         lambda e: do_focus(text_mapping[e.artist]))
    fig.canvas.mpl_connect('button_press_event', on_click)
    fig.canvas.mpl_connect('scroll_event',       lambda e: on_scroll(e, axes_images))

    print('Interface prete.')
    plt.tight_layout()
    plt.show()


def on_scroll(event, axes_images):
    if event.inaxes not in axes_images: return
    ax = event.inaxes
    scale = 1/1.5 if event.button == 'up' else 1.5
    cur_xlim, cur_ylim = ax.get_xlim(), ax.get_ylim()
    new_w = (cur_xlim[1] - cur_xlim[0]) * scale
    new_h = (cur_ylim[1] - cur_ylim[0]) * scale
    ax.set_xlim([event.xdata - new_w/2, event.xdata + new_w/2])
    ax.set_ylim([event.ydata + new_h/2, event.ydata - new_h/2])
    plt.draw()

print('✔  afficher() mise a jour (compatible OCR v7 + fallback Tesseract).')


✔  afficher() mise a jour (compatible OCR v7 + fallback Tesseract).


### `process_and_visualize_v2()` — pipeline complet avec OCR 

Memes etapes 1-3 que l'original + **etape 4 : OCR v7** sur Diagram et Board,
puis `afficher()` alimentee par les resultats OCR.


In [37]:
def process_and_visualize_v2(pdf_path, model_path, output_dir='results'):
    """
    Version etendue de process_and_visualize() :
    - Etapes 1-3 identiques (chargement, conversion PDF, merge, rotations)
    - Etape 4 (nouvelle) : OCR v7 sur Diagram et Board
    - Etape 5 : afficher() alimentee par les resultats OCR v7
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    # Etapes 1-3 : identiques a process_and_visualize()
    try:
        model = YOLO(model_path)
    except Exception as e:
        print(f'Erreur chargement modele : {e}'); return

    print('--- Conversion du PDF en images ---')
    pages = convert_from_path(pdf_path, dpi=300)

    best_images = {'Table': (None, 0), 'Diagram': (None, 0), 'Board': (None, 0)}

    print('--- Extraction des zones via YOLO ---')
    for p_id, page in enumerate(pages):
        results    = model.predict(source=page, conf=0.212, verbose=False)
        page_array = np.array(page)
        for box in results[0].boxes:
            cls_name = results[0].names[int(box.cls[0])]
            key = {'table': 'Table', 'diagram': 'Diagram', 'board': 'Board'}.get(cls_name.lower())
            if not key: continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            margin = 10
            y1_m, y2_m = max(0, y1-margin), min(page_array.shape[0], y2+margin)
            x1_m, x2_m = max(0, x1-margin), min(page_array.shape[1], x2+margin)
            crop = page_array[y1_m:y2_m, x1_m:x2_m]
            if crop is not None and crop.size > 0:
                surface = crop.shape[0] * crop.shape[1]
                if surface > best_images[key][1]:
                    best_images[key] = (crop, surface)

    temp_paths = {}
    for cls in ['Table', 'Diagram', 'Board']:
        img_data, _ = best_images[cls]
        if img_data is None: continue
        img_to_save = cv2.cvtColor(img_data, cv2.COLOR_RGB2BGR)
        if cls == 'Diagram':
            print('Rotation du Schema (90 gauche)...')
            img_to_save = cv2.rotate(img_to_save, cv2.ROTATE_90_COUNTERCLOCKWISE)
        elif cls == 'Board':
            print('Rotation de la Carte (90 droite)...')
            img_to_save = cv2.rotate(img_to_save, cv2.ROTATE_90_CLOCKWISE)
        path = f'temp_{cls.lower()}.png'
        cv2.imwrite(path, img_to_save)
        temp_paths[cls] = path

    if 'Table' not in temp_paths or 'Diagram' not in temp_paths:
        print('\nERREUR : Table ou Diagram manquant.')
        print(f'Trouves : {[k for k, v in best_images.items() if v[0] is not None]}')
        return

    # Etape 4 : OCR v7 sur Diagram et Board
    print('\n--- OCR v7 des etiquettes composants ---')
    # On reutilise le meme modele YOLO pour la detection des etiquettes.
    # Remplacer par YOLO('models/extraction/bestr.pt') si vous avez un modele dedie.
    model_labels = model

    dict_schema = run_ocr_on_image(
        temp_paths['Diagram'], model_labels,
        output_dir=output_dir, label='diagram'
    )
    dict_carte = {}
    if 'Board' in temp_paths:
        dict_carte = run_ocr_on_image(
            temp_paths['Board'], model_labels,
            output_dir=output_dir, label='board'
        )

    # Etape 5 : Interface avec resultats OCR v7
    print('\n--- Affichage ---')
    afficher(
        temp_paths['Table'],
        temp_paths['Diagram'],
        temp_paths.get('Board'),
        ocr_schema=dict_schema,
        ocr_carte=dict_carte,
    )

print('process_and_visualize_v2() definie.')


process_and_visualize_v2() definie.


#### Étape 4 : Lancement de l'interface

`Table` et `Diagram` sont obligatoires pour que l'interface ait du sens. `Board` est optionnel — `.get()` retourne `None` sans lever d'erreur si la clé est absente, et `afficher()` gère ce cas.

On utilise `process_and_visualize_v2()` qui integre l'OCR apres le merge.



In [38]:

# Lancement du pipeline complet
# Modifiez les chemins selon votre configuration

pdf_path_str = next(iter(pdf_path))

PDF_PATH = pdf_path_str
MODEL_PATH = "models/extraction/best.pt"

process_and_visualize(PDF_PATH, MODEL_PATH)
# process_and_visualize_v2(PDF_PATH, MODEL_PATH)


--- Conversion du PDF en images ---


PDFPageCountError: Unable to get page count.
I/O Error: Couldn't open file 'u': No such file or directory.
